## Задача 1
Построить порождающий многочлен для кода Рида-Соломона над полем $GF(2^4)$:
- длина $n=15$
- конструктивное расстояние $d=5$

Объяснение:
- для RS-кода с b=1 порождающий полином:
  g(x)=prod_{i=1}^{d-1}(x-a^i);

In [4]:
PRIM = 0b10011  # p(x)=x^4+x+1


def gf_add(a, b):
    # Сложение в поле GF(2^4) эквивалентно поразрядному XOR двух чисел
    return a ^ b


def gf_mul(a, b):  # Умножение в GF(16) по модулю p(x)=x^4+x+1.
    res = 0
    aa = a
    bb = b
    while bb:
        if bb & 1:  # Если младший бит множителя равен 1
            res ^= aa  # Прибавляем (по XOR) текущий множитель к результату
        bb >>= 1  # Сдвигаем множитель вправо на 1 бит
        aa <<= 1  # Сдвигаем мультипликанд влево на 1 бит (умножаем на x)
        if aa & 0b10000:  # Если после сдвига появился 5-й бит (выход за пределы 4 бит)
            aa ^= PRIM  # Приводим результат по модулю порождающего многочлена (убираем 5-й бит)
    return (
        res & 0b1111
    )  # Возвращаем только младшие 4 бита результата — это элемент GF(16)


def build_tables():
    alpha = 0b0010  # элемент a=x, примитивный элемент поля GF(16)
    alog = [1]  # Список для хранения a^i, начиная с a^0=1
    for _ in range(1, 15):
        # Строим таблицу степеней: каждую следующую степень домножаем на alpha
        alog.append(gf_mul(alog[-1], alpha))
    log = {1: 0}  # Таблица логарифмов: log_{alpha}(value) = index, начнем с log(1)=0
    for i, v in enumerate(alog):
        # Для каждого значения из таблицы степеней запоминаем его логарифм
        log[v] = i
    return alog, log  # Возвращаем обе таблицы: степеней и логарифмов


# Строим таблицы степеней (ALOG) и логарифмов (LOG) для элементов поля GF(16)
# ALOG[i] хранит значение a^i, где a — примитивный элемент поля;
# LOG[v] хранит логарифм по основанию a для каждого ненулевого элемента поля
ALOG, LOG = build_tables()


def gf_to_str(v):
    if v == 0:
        return "0"
    if v == 1:
        return "1"
    return f"a^{LOG[v]}"


def poly_mul_gf(p, q):
    # Умножение полиномов с коэффициентами из GF(16), low->high."""
    res = [0] * (len(p) + len(q) - 1)
    for i, pi in enumerate(p):
        for j, qj in enumerate(q):
            res[i + j] = gf_add(res[i + j], gf_mul(pi, qj))
    return res


def poly_to_str_gf(poly):
    terms = (
        []
    )  # Список для хранения строковых представлений каждого ненулевого члена полинома
    for i, c in enumerate(poly):
        if c == 0:
            continue  # Пропускаем члены с коэффициентом 0
        coef = gf_to_str(c)  # Переводим коэффициент в строку вида "a^k" или "1"
        if i == 0:
            terms.append(f"{coef}")  # Свободный член (x^0), просто коэффициент
        elif i == 1:
            terms.append(f"{coef}*x")  # Член при x^1
        else:
            terms.append(f"{coef}*x^{i}")  # Член при x^i для i>1
    return (
        " + ".join(terms) if terms else "0"
    )  # Склеиваем члены через " + " или возвращаем "0", если все коэффициенты нули


def rs_generator_poly(n, d, b=1):

    # g(x)=prod_{j=b}^{b+d-2}(x-a^j).
    # Здесь n=15, поле GF(16), обычно b=1 (как в конспекте).

    g = [1]
    for j in range(b, b + d - 1):
        # x - a^j, в char2 это x + a^j => [a^j, 1]
        g = poly_mul_gf(g, [ALOG[j % n], 1])
    return g


def eval_poly_at_alpha_power(coeffs, j, n=15):
    # Вычисляет v(a^j), где v(x)=sum coeffs[i] x^i, coeffs low->high.
    s = 0
    for i, c in enumerate(coeffs):
        if c:
            s = gf_add(s, gf_mul(c, ALOG[(i * j) % n]))
    return s


def task1():
    print("Задача 1")
    n = 15
    d = 5
    g = rs_generator_poly(n=n, d=d, b=1)
    print("Для RS-кода над GF(16): n=15, d=5, b=1")
    print("g(x) = (x-a)(x-a^2)(x-a^3)(x-a^4)")
    print("g(x) =", poly_to_str_gf(g))
    print("deg g =", len(g) - 1)
    print("k = n - deg g =", n - (len(g) - 1))
    print()
    return g

In [5]:
task1()

Задача 1
Для RS-кода над GF(16): n=15, d=5, b=1
g(x) = (x-a)(x-a^2)(x-a^3)(x-a^4)
g(x) = a^10 + a^3*x + a^6*x^2 + a^13*x^3 + 1*x^4
deg g = 4
k = n - deg g = 11



[7, 8, 12, 13, 1]

## Задача 2
Рассмотрим РС-код $(15, 9)$ над полем $GF(2^4)$, исправляющий 3 ошибки.
Поле задано примитивным многочленом $p(x)=1+x+x^4$.
Получено слово
$$
v(x)=a+a^3x+a^{10}x^2+a^{12}x^3+a^{13}x^4+a^3x^5+a^9x^6+a^5x^7+x^8+a^{10}x^9+a^8x^{10}+ax^{11}+ax^{12}+a^{13}x^{13}+a^2x^{14}.
$$
Нужно определить вектор ошибок $e(x)$.

Объяснение:
- для RS(15,9): d=n-k+1=7 и t=3;
- по алгоритму из конспекта вычисляем синдромы S1..S6;
- если все синдромы нулевые, то ошибок нет и e(x)=0.

In [6]:
def task2():
    print("Задача 2")
    # коэффициенты v(x) от x^0 до x^14 (по условию)
    exp_by_power = {
        0: 1,
        1: 3,
        2: 10,
        3: 12,
        4: 13,
        5: 3,
        6: 9,
        7: 5,
        8: 0,  # здесь коэффициент просто 1
        9: 10,
        10: 8,
        11: 1,
        12: 1,
        13: 13,
        14: 2,
    }
    v = []
    for i in range(15):
        e = exp_by_power[i]
        v.append(1 if e == 0 else ALOG[e])

    print("Принятый полином v(x) задан коэффициентами в степенях a.")
    print("Считаем синдромы для RS(15,9): S1..S6 (так как d=7, t=3).")

    synd = []
    # Вычисляем синдромы S1..S6: подставляем alpha^j в v(x) для j=1..6
    for j in range(1, 7):
        sj = eval_poly_at_alpha_power(v, j)  # sj = v(alpha^j)
        synd.append(sj)

    # Выводим вычисленные синдромы в удобочитаемом виде
    for idx, sj in enumerate(synd, start=1):
        print(f"  S{idx} = {gf_to_str(sj)}")

    # Проверяем: если все синдромы равны нулю — ошибок нет
    if all(s == 0 for s in synd):
        print("\nВсе синдромы нулевые => ошибок нет.")
        print("Следовательно, вектор ошибок:")
        print("  e(x) = 0")
    else:
        # На случай, если в другом варианте синдромы окажутся ненулевыми.
        # Тогда необходимо полное декодирование (по методу PGZ: определение позиций и величин ошибок)
        print(
            "\nСиндромы ненулевые: нужно полное декодирование PGZ (локаторы/величины ошибок)."
        )


task2()

Задача 2
Принятый полином v(x) задан коэффициентами в степенях a.
Считаем синдромы для RS(15,9): S1..S6 (так как d=7, t=3).
  S1 = 0
  S2 = 0
  S3 = 0
  S4 = 0
  S5 = 0
  S6 = 0

Все синдромы нулевые => ошибок нет.
Следовательно, вектор ошибок:
  e(x) = 0
